In [1]:
import sys
sys.path.append("../")

from tools import download_documentation_data

data = download_documentation_data()

interesting_pages = []

for page in data:
    if len(page["content"]) <= 1000:
        continue

    if page["filename"] == "changelog/changelog.mdx":
        continue

    interesting_pages.append(page)

doc = interesting_pages[15]
content = doc["content"]

print("Document:", doc["filename"])
print("Content length:", len(content))
print(content[:700])

Document: docs/platform/datasets_overview.mdx
Content length: 2257
<Check>
  Datasets are available in **Evidently OSS, Cloud** and **Evidently Enterprise**.
</Check>

## What is a Dataset?

**Datasets** are collections of data from your application used for analysis and automated checks. You can bring in existing datasets, capture live data, or create synthetic datasets.

![](/images/dataset_llm.png)

## How to create a Dataset?

You can add Datasets to the platform in multiple ways:

- **Upload directly**. Use the UI (Evidently Cloud) to upload CSV files or push datasets via the Python API.
- **Upload with Reports**. Attach datasets to Reports when running local evaluations. This is optional — you can also upload only summary metrics.
- **Generate synthet


In [2]:
from pydantic import BaseModel, Field
from typing import Literal


class GeneratedQuestion(BaseModel):
    user_question: str = Field(
        description="the question we generate"
    )

    reference_answer: str = Field(
        description="the correct answer based on the document"
    )

    line_number_start: int = Field(
        description="starting line of the reference answer"
    )

    line_number_end: int = Field(
        description="ending line of the reference answer"
    )

    question_type: Literal[
        "frustrated_beginner",
        "hurried_developer",
        "context_assumer",
        "direct_inquirer"
    ]


class QuestionsResponse(BaseModel):
    questions: list[GeneratedQuestion]


class GeneratedQuestions(BaseModel):
    questions: list[GeneratedQuestion]
    filename: str


print("Structured Output models ready")

Structured Output models ready


In [3]:
instructions = """
You are an expert at generating realistic user questions based on technical documentation.
Your task is to analyze the provided documentation page and generate questions that a user might realistically ask.

We want to generate questions that represent REAL users who are stuck, frustrated, or searching for specific information — rather than perfectly phrased textbook questions.

When generating questions, randomly adopt one of the following personas/styles for each question:

1. The Frustrated Beginner: Uses non-technical terms, explains what they are trying to achieve (their goal) but are stuck.
2. The Hurried Developer: Uses fragments, keywords, or short phrases instead of full sentences.
3. The Context-Assumer: Asks a question assuming the bot knows what page they are looking at or what step they are on.
4. The Direct Inquirer: Asks a straightforward, somewhat technical question in a conversational tone.

For each generated question, you must also extract the reference answer directly based on the document's content.
You must also pinpoint the exact starting and ending line numbers in the provided document where this answer is located.

Follow these strict guidelines:
1. DO NOT end questions with phrases like "based on this text" or "according to the documentation."
2. DO NOT make the questions too formal or academic.
3. The reference answer should be comprehensive and directly supported by the text.
4. The line numbers must accurately reflect exactly where the reference answer can be found in the provided document with line numbers attached.
""".strip()

print("Generator instructions ready")

Generator instructions ready


In [4]:
from pydantic_ai import Agent

data_generator = Agent(
    name="data_generator",
    model="openai:gpt-4o-mini",
    instructions=instructions,
    output_type=QuestionsResponse
)

print("Data generator ready")

Data generator ready


In [5]:
def add_line_numbers(document: str) -> str:
    lines = document.splitlines()

    numbered_lines = [
        f"{i+1}: {line}"
        for i, line in enumerate(lines)
    ]

    return "\n".join(numbered_lines)


numbered_content = add_line_numbers(content)

print(numbered_content[:500])

1: <Check>
2:   Datasets are available in **Evidently OSS, Cloud** and **Evidently Enterprise**.
3: </Check>
4: 
5: ## What is a Dataset?
6: 
7: **Datasets** are collections of data from your application used for analysis and automated checks. You can bring in existing datasets, capture live data, or create synthetic datasets.
8: 
9: ![](/images/dataset_llm.png)
10: 
11: ## How to create a Dataset?
12: 
13: You can add Datasets to the platform in multiple ways:
14: 
15: - **Upload directly**. Us


In [6]:
prompt_template = """
Number of questions:
{number_of_questions}

Document:
{numbered_content}
""".strip()


def create_user_prompt(document_content: str) -> str:
    number_of_questions = len(document_content) // 1000
    numbered_content = add_line_numbers(document_content)

    prompt = prompt_template.format(
        number_of_questions=number_of_questions,
        numbered_content=numbered_content
    )

    return prompt.strip()


user_prompt = create_user_prompt(content)

print(user_prompt[:350])
print("Requested questions:", len(content) // 1000)

Number of questions:
2

Document:
1: <Check>
2:   Datasets are available in **Evidently OSS, Cloud** and **Evidently Enterprise**.
3: </Check>
4: 
5: ## What is a Dataset?
6: 
7: **Datasets** are collections of data from your application used for analysis and automated checks. You can bring in existing datasets, capture live data, or create synthet
Requested questions: 2


In [7]:
questions_result = await data_generator.run(user_prompt)

questions = questions_result.output

print("Generated questions:", len(questions.questions))

for q in questions.questions:
    print("\nQuestion:", q.user_question)
    print("Reference answer:", q.reference_answer)
    print("Source lines:", q.line_number_start, "-", q.line_number_end)
    print("Question type:", q.question_type)
    print("-" * 50)

Generated questions: 2

Question: I'm trying to figure out how to create a dataset, but I feel lost. What are my options?
Reference answer: You can add Datasets to the platform in multiple ways:

- **Upload directly**. Use the UI (Evidently Cloud) to upload CSV files or push datasets via the Python API.
- **Upload with Reports**. Attach datasets to Reports when running local evaluations. This is optional — you can also upload only summary metrics.
- **Generate synthetic data**. Use built-in platform features to generate synthetic evaluation datasets. (Cloud only).
- **Create from Traces**. During tracing, Evidently automatically generates tabular datasets that can be used for evaluations. (Cloud only).
Source lines: 13 - 18
Question type: frustrated_beginner
--------------------------------------------------

Question: I need to generate synthetic data for tests but only on Cloud, right?
Reference answer: You can synthesize evaluation datasets directly in Evidently Cloud:

- **Generate

In [8]:
lines = content.splitlines()

for q in questions.questions:
    print("\nQuestion:", q.user_question)
    print("Reference lines:")

    for i in range(q.line_number_start - 1, q.line_number_end):
        print(f"{i+1}: {lines[i]}")

    print("-" * 50)


Question: I'm trying to figure out how to create a dataset, but I feel lost. What are my options?
Reference lines:
13: You can add Datasets to the platform in multiple ways:
14: 
15: - **Upload directly**. Use the UI (Evidently Cloud) to upload CSV files or push datasets via the Python API.
16: - **Upload with Reports**. Attach datasets to Reports when running local evaluations. This is optional — you can also upload only summary metrics.
17: - **Generate synthetic data**. Use built-in platform features to generate synthetic evaluation datasets. (Cloud only).
18: - **Create from Traces**. During tracing, Evidently automatically generates tabular datasets that can be used for evaluations. (Cloud only).
--------------------------------------------------

Question: I need to generate synthetic data for tests but only on Cloud, right?
Reference lines:
26: You can synthesize evaluation datasets directly in Evidently Cloud:
27: 
28: - **Generate from examples or description**. Describe spec

In [9]:
generated_question = GeneratedQuestions(
    questions=questions.questions,
    filename=doc["filename"]
)

print("Source:", generated_question.filename)
print("Questions:", len(generated_question.questions))

Source: docs/platform/datasets_overview.mdx
Questions: 2


In [10]:
total_questions = sum(
    len(page["content"]) // 1000
    for page in interesting_pages
)

largest_page = max(
    interesting_pages,
    key=lambda page: len(page["content"])
)

print("Documents:", len(interesting_pages))
print("Requested questions:", total_questions)
print("Largest document:", largest_page["filename"])
print("Largest document characters:", len(largest_page["content"]))

Documents: 71
Requested questions: 549
Largest document: metrics/all_metrics.mdx
Largest document characters: 54996


In [12]:
from tqdm.auto import tqdm

pilot_results = []

for doc in tqdm(interesting_pages[:3]):
    content = doc["content"]

    user_prompt = create_user_prompt(content)
    result = await data_generator.run(user_prompt)

    generated = GeneratedQuestions(
        questions=result.output.questions,
        filename=doc["filename"]
    )

    pilot_results.append(generated)

    print(
        doc["filename"],
        "→",
        len(generated.questions),
        "questions"
    )

print(
    "Total generated:",
    sum(len(item.questions) for item in pilot_results)
)

  0%|          | 0/3 [00:00<?, ?it/s]

docs/library/data_definition.mdx → 11 questions
docs/library/descriptors.mdx → 12 questions
docs/library/evaluations_overview.mdx → 3 questions
Total generated: 26


In [13]:
import pandas as pd

rows = []

for gqs in pilot_results:
    for q in gqs.questions:
        rows.append({
            "question": q.user_question,
            "reference_answer": q.reference_answer,
            "line_number_start": q.line_number_start,
            "line_number_end": q.line_number_end,
            "question_type": q.question_type,
            "filename": gqs.filename
        })

df = pd.DataFrame(rows)

df.to_csv("questions_generated_pilot.csv", index=False)

print("Saved questions:", len(df))
print("Columns:", list(df.columns))

Saved questions: 26
Columns: ['question', 'reference_answer', 'line_number_start', 'line_number_end', 'question_type', 'filename']


In [14]:
from tqdm.auto import tqdm
import pandas as pd

all_rows = df.to_dict(orient="records")
errors = []

for doc in tqdm(interesting_pages[3:]):
    try:
        user_prompt = create_user_prompt(doc["content"])
        result = await data_generator.run(user_prompt)

        for q in result.output.questions:
            all_rows.append({
                "question": q.user_question,
                "reference_answer": q.reference_answer,
                "line_number_start": q.line_number_start,
                "line_number_end": q.line_number_end,
                "question_type": q.question_type,
                "filename": doc["filename"]
            })

        pd.DataFrame(all_rows).to_csv(
            "questions_generated_partial.csv",
            index=False
        )

    except Exception as exc:
        errors.append((doc["filename"], str(exc)))
        print("Failed:", doc["filename"], str(exc)[:200])

print("Total generated:", len(all_rows))
print("Failed documents:", len(errors))

for filename, error in errors:
    print(filename, error[:200])

  0%|          | 0/68 [00:00<?, ?it/s]

Total generated: 487
Failed documents: 0


In [15]:
import pandas as pd

final_df = pd.read_csv("questions_generated_partial.csv")

print("Total questions:", len(final_df))
print("Source documents:", final_df["filename"].nunique())
print("Missing reference answers:",
      final_df["reference_answer"].isna().sum())

final_df.to_csv("questions_generated.csv", index=False)

print("Saved: questions_generated.csv")

Total questions: 487
Source documents: 71
Missing reference answers: 0
Saved: questions_generated.csv
